In [ ]:
library(harmony)
library(hdf5r)
library(Seurat)
library(dplyr)

x <- Read10X_h5("/path/to/nature_study/h5/H_5_A-filtered_feature_bc_matrix.h5")
names(x)   # in multiome this will usually expose the assay entries; you want the RNA one

In [ ]:
suppressPackageStartupMessages({
  library(hdf5r)
  library(Seurat)
  library(Matrix)
  library(dplyr)
  library(purrr)
  library(stringr)
  library(tibble)
  library(ggplot2)
})

set.seed(1234)

h5_dir   <- "/path/to/nature_study/h5"
proj_dir <- "/path/to/nature_study/ArchR"
out_dir  <- file.path(proj_dir, "RNA_annotations")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
sample_map <- tribble(
  ~sample,       ~donor,     ~condition,
  "H_5_A",       "H_5",      "disomy",
  "H_5_B",       "H_5",      "disomy",
  "H_6_A",       "H_6",      "disomy",
  "H_7_A",       "H_7",      "disomy",
  "H_7_B",       "H_7",      "disomy",
  "H_7_C",       "H_7",      "disomy",
  "TS21_4_H",    "TS21_4",   "Ts21",
  "TS21_4_I",    "TS21_4",   "Ts21",
  "TS21_11_F",   "TS21_11",  "Ts21",
  "TS21_11_G",   "TS21_11",  "Ts21",
  "TS21_16_A",   "TS21_16",  "Ts21",
  "TS21_16_B",   "TS21_16",  "Ts21"
) %>%
  mutate(h5 = file.path(h5_dir, paste0(sample, "-filtered_feature_bc_matrix.h5")))

stopifnot(all(file.exists(sample_map$h5)))
sample_map

In [ ]:
extract_rna_counts <- function(h5_file) {
  x <- Read10X_h5(h5_file, use.names = TRUE, unique.features = TRUE)

  if (inherits(x, "dgCMatrix")) {
    return(x)
  }

  if (is.list(x)) {
    return(x[["Gene Expression"]])
  }

  stop("Unexpected object returned by Read10X_h5: ", paste(class(x), collapse = ", "))
}

In [ ]:
test_counts <- extract_rna_counts(sample_map$h5[1])

dim(test_counts)
head(colnames(test_counts))
head(rownames(test_counts))

In [ ]:
obj_list <- lapply(seq_len(nrow(sample_map)), function(i) {
  meta_row <- sample_map[i, ]

  counts <- extract_rna_counts(meta_row$h5)

  # Critical: match ArchR naming convention
  colnames(counts) <- paste(meta_row$sample, colnames(counts), sep = "#")

  obj <- CreateSeuratObject(
    counts = counts,
    assay = "RNA",
    project = "EMTAB13070_RNA",
    min.cells = 3,
    min.features = 50
  )

  obj$sample    <- meta_row$sample
  obj$donor     <- meta_row$donor
  obj$condition <- meta_row$condition
  obj$barcode   <- sub(paste0("^", meta_row$sample, "#"), "", colnames(obj))
  obj$archr_cell <- colnames(obj)

  obj$percent.mt <- PercentageFeatureSet(obj, pattern = "^MT-")

  hb_genes <- intersect(
    rownames(obj),
    c("HBA1","HBA2","HBB","HBD","HBE1","HBM","HBQ1","HBZ")
  )

  obj$percent.hb <- if (length(hb_genes) > 0) {
    PercentageFeatureSet(obj, features = hb_genes)
  } else {
    0
  }

  obj
})

names(obj_list) <- sample_map$sample
names(obj_list)

In [ ]:
rna <- merge(
  x = obj_list[[1]],
  y = obj_list[-1],
  project = "EMTAB13070_RNA"
)

rna
table(rna$condition)
table(rna$sample)

In [ ]:
rna <- subset(
  rna,
  subset =
    nFeature_RNA >= 100 &
    nCount_RNA   >= 200 &
    percent.mt   <= 25
)

rna

In [ ]:
qc_df <- rna@meta.data %>%
  rownames_to_column("cell") %>%
  count(sample, donor, condition, name = "n_cells")

write.table(
  qc_df,
  file = file.path(out_dir, "RNA_QC_cells_per_sample.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

qc_df

In [ ]:
DefaultAssay(rna) <- "RNA"

rna <- NormalizeData(rna, normalization.method = "LogNormalize", scale.factor = 1e4, verbose = FALSE)
rna <- FindVariableFeatures(rna, selection.method = "vst", nfeatures = 3000, verbose = FALSE)
rna <- ScaleData(rna, features = VariableFeatures(rna), verbose = FALSE)
rna <- RunPCA(rna, features = VariableFeatures(rna), npcs = 50, verbose = FALSE)

ElbowPlot(rna, ndims = 50)

In [ ]:
DimPlot(rna, reduction = "pca", group.by = "sample")
DimPlot(rna, reduction = "pca", group.by = "donor")
DimPlot(rna, reduction = "pca", group.by = "condition")

In [ ]:
rna <- RunHarmony(
  object = rna,
  group.by.vars = "sample",
  reduction.use = "pca",
  dims.use = 1:30,
  assay.use = "RNA",
  verbose = FALSE
)

reduction_use <- "harmony"

In [ ]:
Reductions(rna)
DimPlot(rna, reduction = "harmony", group.by = "sample")
DimPlot(rna, reduction = "harmony", group.by = "condition")

In [ ]:
rna <- RunUMAP(rna, reduction = reduction_use, dims = 1:30, verbose = FALSE)
rna <- FindNeighbors(rna, reduction = reduction_use, dims = 1:30, verbose = FALSE)
rna <- FindClusters(rna, resolution = 0.5, verbose = FALSE)

DimPlot(rna, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)

In [ ]:
DefaultAssay(rna) <- "RNA"

# inspect current layers
Layers(rna[["RNA"]])

# join layers for DE
rna <- JoinLayers(rna, assay = "RNA")

# check again
Layers(rna[["RNA"]])

Idents(rna) <- "seurat_clusters"

markers <- FindAllMarkers(
  object = rna,
  assay = "RNA",
  only.pos = TRUE,
  test.use = "wilcox",
  min.pct = 0.10,
  logfc.threshold = 0.25,
  return.thresh = 0.05
)

# safe check
dim(markers)
head(markers)

top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10, with_ties = FALSE) %>%
  ungroup()

top10

In [ ]:
# ------------------------------------------------------------
# 2) UMAP / graph / clustering
# ------------------------------------------------------------
rna <- RunUMAP(rna, reduction = reduction_use, dims = 1:30, verbose = FALSE)
rna <- FindNeighbors(rna, reduction = reduction_use, dims = 1:30, verbose = FALSE)
rna <- FindClusters(rna, resolution = 0.5, verbose = FALSE)

DimPlot(rna, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)
DimPlot(rna, reduction = "umap", group.by = "sample", raster = TRUE)
DimPlot(rna, reduction = "umap", group.by = "condition", raster = TRUE)

In [ ]:
# Optional if 0.5 is too coarse for B-lineage states:
#rna <- FindClusters(rna, resolution = 0.8, verbose = FALSE)
#DimPlot(rna, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# 3) Cluster markers
# ------------------------------------------------------------
DefaultAssay(rna) <- "RNA"
Idents(rna) <- "seurat_clusters"

markers <- FindAllMarkers(
  object = rna,
  assay = "RNA",
  only.pos = TRUE,
  test.use = "wilcox",
  min.pct = 0.10,
  logfc.threshold = 0.25,
  return.thresh = 0.05
)

top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10, with_ties = FALSE) %>%
  ungroup()

top10

In [ ]:
broad_map <- c(
  "0"  = "unresolved_contam_like",
  "1"  = "erythroid_like",
  "2"  = "NK",
  "23" = "low_complexity",
  "24" = "megakaryocyte_platelet",
  "25" = "hepatocyte_like"
)

rna$broad_annot <- unname(broad_map[as.character(rna$seurat_clusters)])
rna$broad_annot[is.na(rna$broad_annot)] <- "unassigned"

table(rna$seurat_clusters, rna$broad_annot)
DimPlot(rna, reduction = "umap", group.by = "broad_annot", label = TRUE, raster = TRUE)

In [ ]:
#panel <- c(
  # stem / upstream progenitors
#  "CD34","SPINK2","HLF","MECOM","FLT3","IL7R","GATA2","SATB1",

  # B-lineage priming / progenitors
#  "EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A","CD79B","MS4A1","CD74",

  # erythroid / heme
#  "GATA1","KLF1","ALAS2","AHSP","HBA1","HBA2","HBB","GYPA","HMBS","FECH",

  # interferon
#  "IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","IFI44L",

  # myeloid / mono / DC
#  "LYZ","MPO","ELANE","AZU1","S100A8","S100A9","FCER1A","CST3","CTSS",

  # T / NK
#  "TRAC","TRBC1","NKG7","GNLY","KLRD1"
#)

#DotPlot(rna, features = panel, group.by = "seurat_clusters") + RotatedAxis()

#FeaturePlot(
#  rna,
#  features = c("EBF1","PAX5","CD24","DNTT","HBB","ALAS2","GATA1","IFI6","MX1"),
#  reduction = "umap",
#  raster = TRUE,
#  ncol = 3
#)

In [ ]:
# ------------------------------------------------------------
# 4) Broad cleanup before progenitor refinement
# ------------------------------------------------------------
Idents(rna) <- "seurat_clusters"

# remove obvious contaminants / weak clusters from the refinement pass
drop1 <- c("0", "23", "25")  # unresolved_contam_like, low_complexity, hepatocyte_like

rna_h <- subset(
  rna,
  idents = setdiff(levels(Idents(rna)), drop1)
)

DimPlot(rna_h, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)
DimPlot(rna_h, reduction = "umap", group.by = "broad_annot", label = TRUE, raster = TRUE)

In [ ]:
#Idents(rna) <- "seurat_clusters"

#drop1 <- c("0", "23", "25")
#rna_h <- subset(rna, idents = setdiff(levels(Idents(rna)), drop1))

#DimPlot(rna_h, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# 5) Select candidate progenitor / B-lineage clusters
# ------------------------------------------------------------
# Replace these IDs after checking the DotPlot and FeaturePlot from Cell 19
prog_clusters <- c("3","4","5","6","7","8","9","10")

rna_prog <- subset(rna_h, idents = prog_clusters)

table(rna_prog$seurat_clusters)
DimPlot(rna_prog, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)

In [ ]:
# example only: replace with your real progenitor/B-lineage cluster IDs
#prog_clusters <- c("3","4","5","6","7","8","9","10")

#rna_prog <- subset(rna_h, idents = prog_clusters)

In [ ]:
# ------------------------------------------------------------
# 6) Re-cluster the progenitor subset
# ------------------------------------------------------------
DefaultAssay(rna_prog) <- "RNA"
rna_prog <- JoinLayers(rna_prog, assay = "RNA")

rna_prog <- NormalizeData(rna_prog, verbose = FALSE)
rna_prog <- FindVariableFeatures(rna_prog, verbose = FALSE)
rna_prog <- ScaleData(rna_prog, verbose = FALSE)
rna_prog <- RunPCA(rna_prog, verbose = FALSE)

rna_prog <- RunHarmony(
  object = rna_prog,
  group.by.vars = "sample",
  reduction.use = "pca",
  dims.use = 1:30,
  verbose = FALSE
)

rna_prog <- RunUMAP(rna_prog, reduction = "harmony", dims = 1:30, verbose = FALSE)
rna_prog <- FindNeighbors(rna_prog, reduction = "harmony", dims = 1:30, verbose = FALSE)
rna_prog <- FindClusters(rna_prog, resolution = 0.3, verbose = FALSE)

DimPlot(rna_prog, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)
DimPlot(rna_prog, reduction = "umap", group.by = "condition", raster = TRUE)
DimPlot(rna_prog, reduction = "umap", group.by = "donor", raster = TRUE)

Idents(rna_prog) <- "seurat_clusters"

prog_markers <- FindAllMarkers(
  object = rna_prog,
  assay = "RNA",
  only.pos = TRUE,
  test.use = "wilcox",
  min.pct = 0.10,
  logfc.threshold = 0.25,
  return.thresh = 0.05
)

prog_top10 <- prog_markers %>%
  dplyr::group_by(cluster) %>%
  dplyr::slice_max(order_by = avg_log2FC, n = 10, with_ties = FALSE) %>%
  dplyr::ungroup()

print(prog_top10, n = 200)

In [ ]:
# ------------------------------------------------------------
# 7) Score the refined progenitor subset
# ------------------------------------------------------------
b_priming_genes <- list(c("EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A"))
ery_heme_genes  <- list(c("GATA1","KLF1","ALAS2","AHSP","HBA1","HBA2","HBB","HMBS","FECH"))
ifn_genes       <- list(c("IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","IFI44L"))
stem_genes      <- list(c("CD34","SPINK2","HLF","FLT3","GATA2","MECOM"))

rna_prog <- AddModuleScore(rna_prog, features = b_priming_genes, name = "Bpriming")
rna_prog <- AddModuleScore(rna_prog, features = ery_heme_genes,  name = "EryHeme")
rna_prog <- AddModuleScore(rna_prog, features = ifn_genes,       name = "IFN")
rna_prog <- AddModuleScore(rna_prog, features = stem_genes,      name = "StemCoreProg")

DotPlot(
  rna_prog,
  features = c(
    "CD34","SPINK2","HLF","FLT3","GATA2","MECOM",
    "EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A",
    "GATA1","KLF1","ALAS2","AHSP","HBB",
    "IFI6","IFIT1","ISG15","MX1",
    "LYZ","MPO","S100A8","S100A9"
  ),
  group.by = "seurat_clusters"
) + RotatedAxis()

FeaturePlot(
  rna_prog,
  features = c("StemCoreProg1","Bpriming1","EryHeme1","IFN1"),
  reduction = "umap",
  raster = TRUE,
  ncol = 2
)

In [ ]:
#DefaultAssay(rna_prog) <- "RNA"

#rna_prog <- FindVariableFeatures(rna_prog, verbose = FALSE)
#rna_prog <- ScaleData(rna_prog, verbose = FALSE)
#rna_prog <- RunPCA(rna_prog, verbose = FALSE)

#rna_prog <- RunHarmony(
#  object = rna_prog,
#  group.by.vars = "sample",
#  reduction.use = "pca",
#  dims.use = 1:30,
#  verbose = FALSE
#)

#rna_prog <- RunUMAP(rna_prog, reduction = "harmony", dims = 1:30, verbose = FALSE)
#rna_prog <- FindNeighbors(rna_prog, reduction = "harmony", dims = 1:30, verbose = FALSE)
#rna_prog <- FindClusters(rna_prog, resolution = 0.4, verbose = FALSE)

In [ ]:
# ------------------------------------------------------------
# 8) Write refined progenitor labels back into full rna object
# ------------------------------------------------------------

# First: annotate the refined progenitor clusters manually.
# Replace this example mapping with your real labels after reviewing prog_top10 and plots.
cluster_to_celltype_prog <- c(
  "0" = "erythroid_like_2",
  "1" = "NK_like",
  "2" = "megakaryocyte_like",
  "3" = "HSC_MPP",
  "4" = "B_primed_prog",
  "5" = "myeloid_DC_like",
  "6" = "mixed_unresolved",
  "7" = "granulocyte_prog"
)
rna_prog$celltype_refined <- unname(
  cluster_to_celltype_prog[as.character(rna_prog$seurat_clusters)]
)

# Define real stem_core from the refined progenitor labels
stem_core_labels <- c("HSC_MPP")

rna_prog$stem_core_refined <- ifelse(
  rna_prog$celltype_refined %in% stem_core_labels,
  "stem_core",
  "other"
)

# Start from the broad labels for the full object
rna$celltype_rna <- as.character(rna$broad_annot)
rna$stem_core <- "other"

# Overwrite the progenitor subset with refined labels
rna$celltype_rna[colnames(rna_prog)] <- rna_prog$celltype_refined
rna$stem_core[colnames(rna_prog)] <- rna_prog$stem_core_refined

rna$stem_core <- factor(rna$stem_core, levels = c("stem_core", "other"))

table(rna$celltype_rna, useNA = "ifany")
table(rna$stem_core, useNA = "ifany")
DimPlot(rna, reduction = "umap", group.by = "celltype_rna", label = TRUE, raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# Update full-rna module scores without overwriting refined labels
# ------------------------------------------------------------
DefaultAssay(rna) <- "RNA"

b_priming_genes <- list(c("EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A"))
ery_heme_genes  <- list(c("GATA1","KLF1","ALAS2","AHSP","HBA1","HBA2","HBB","HMBS","FECH"))
ifn_genes       <- list(c("IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","IFI44L"))

# remove old scores if present
for (nm in c("Bpriming1", "EryHeme1", "IFN1")) {
  if (nm %in% colnames(rna[[]])) {
    rna[[nm]] <- NULL
  }
}

rna <- AddModuleScore(rna, features = b_priming_genes, name = "Bpriming")
rna <- AddModuleScore(rna, features = ery_heme_genes,  name = "EryHeme")
rna <- AddModuleScore(rna, features = ifn_genes,       name = "IFN")

# sanity checks
table(rna$celltype_rna, useNA = "ifany")
table(rna$stem_core, useNA = "ifany")
summary(rna$Bpriming1)
summary(rna$EryHeme1)
summary(rna$IFN1)

In [ ]:
library(ArchR)

archr_proj <- loadArchRProject(
  "/path/to/nature_study/ArchR/ArchRProject_saved_withRNA"
)

class(archr_proj)
length(getCellNames(archr_proj))

In [ ]:
library(tibble)
library(dplyr)

# Build annotation table directly from Seurat metadata
anno <- rna[[]] %>%
  rownames_to_column("cellNames") %>%
  transmute(
    cellNames,
    sample,
    donor,
    condition,
    seurat_clusters = as.character(seurat_clusters),
    broad_annot = as.character(broad_annot),
    rna_celltype = as.character(celltype_rna),
    stem_core = as.character(stem_core),
    Bpriming1,
    EryHeme1,
    IFN1
  )

# Check overlap with ArchR
archr_cells <- getCellNames(archr_proj)
common <- intersect(anno$cellNames, archr_cells)

length(anno$cellNames)
length(archr_cells)
length(common)

anno_use <- anno[match(common, anno$cellNames), , drop = FALSE]

# Add / overwrite columns into ArchR
archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$broad_annot,
  name = "broad_annot",
  cells = anno_use$cellNames,
  force = TRUE
)

archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$rna_celltype,
  name = "rna_celltype",
  cells = anno_use$cellNames,
  force = TRUE
)

archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$stem_core,
  name = "stem_core",
  cells = anno_use$cellNames,
  force = TRUE
)

archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$Bpriming1,
  name = "Bpriming1",
  cells = anno_use$cellNames,
  force = TRUE
)

archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$EryHeme1,
  name = "EryHeme1",
  cells = anno_use$cellNames,
  force = TRUE
)

archr_proj <- addCellColData(
  ArchRProj = archr_proj,
  data = anno_use$IFN1,
  name = "IFN1",
  cells = anno_use$cellNames,
  force = TRUE
)

In [ ]:
head(getCellColData(archr_proj)[, c("Sample", "donor", "condition", "broad_annot", "rna_celltype", "stem_core")])
table(archr_proj$broad_annot, useNA = "ifany")
table(archr_proj$rna_celltype, useNA = "ifany")
table(archr_proj$stem_core, useNA = "ifany")

In [ ]:
archr_proj <- saveArchRProject(
  ArchRProj = archr_proj,
  outputDirectory = "/path/to/nature_study/ArchR/ArchRProject_saved_refinedRNA",
  load = TRUE
)

In [ ]:
# Optional: standalone refined RNA annotation table
dir.create("/path/to/nature_study/ArchR/RNA_annotations",
           recursive = TRUE, showWarnings = FALSE)

write.table(
  anno,
  "/path/to/nature_study/ArchR/RNA_annotations/rna_annotations_refined_for_ArchR.tsv",
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

In [ ]:
library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)

# ------------------------------------------------------------
# A) donor-aware program shift inside stem_core
# ------------------------------------------------------------
stopifnot(all(c(
  "celltype_rna", "stem_core", "donor", "condition",
  "Bpriming1", "EryHeme1", "IFN1"
) %in% colnames(rna[[]])))

stem_df <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(
    stem_core == "stem_core",
    !is.na(donor),
    !is.na(condition)
  ) %>%
  tibble::as_tibble()

# check how many stem_core cells per donor
stem_counts <- stem_df %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_stem_core_cells = dplyr::n(),
    .groups = "drop"
  )

stem_counts

# donor-level mean program scores
stem_prog <- stem_df %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_cells   = dplyr::n(),
    Bpriming1 = mean(Bpriming1, na.rm = TRUE),
    EryHeme1  = mean(EryHeme1,  na.rm = TRUE),
    IFN1      = mean(IFN1,      na.rm = TRUE),
    .groups = "drop"
  )

stem_prog

stem_prog_long <- stem_prog %>%
  tidyr::pivot_longer(
    cols = c(Bpriming1, EryHeme1, IFN1),
    names_to = "program",
    values_to = "score"
  )

ggplot(stem_prog_long, aes(x = condition, y = score)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  facet_wrap(~program, scales = "free_y") +
  theme_bw()

stem_prog_tests <- stem_prog_long %>%
  dplyr::group_by(program) %>%
  dplyr::summarise(
    disomy_mean = mean(score[condition == "disomy"]),
    ts21_mean   = mean(score[condition == "Ts21"]),
    delta       = ts21_mean - disomy_mean,
    p_wilcox    = wilcox.test(score ~ condition, exact = FALSE)$p.value,
    .groups = "drop"
  )

stem_prog_tests

In [ ]:
# ------------------------------------------------------------
# B) donor-aware gene-level check inside stem_core
# ------------------------------------------------------------
genes_b   <- c("EBF1", "PAX5", "CD24", "DNTT")
genes_ery <- c("GATA1", "KLF1", "ALAS2", "AHSP", "HBB")
genes_ifn <- c("IFI6", "IFIT1", "ISG15", "MX1")
genes_use <- c(genes_b, genes_ery, genes_ifn)

stem_expr <- FetchData(
  object = rna,
  vars = c("stem_core", "donor", "condition", genes_use)
) %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(stem_core == "stem_core") %>%
  tibble::as_tibble()

stem_gene_donor <- stem_expr %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    dplyr::across(dplyr::all_of(genes_use), ~mean(.x, na.rm = TRUE)),
    .groups = "drop"
  )

stem_gene_long <- stem_gene_donor %>%
  tidyr::pivot_longer(
    cols = dplyr::all_of(genes_use),
    names_to = "gene",
    values_to = "expr"
  )

ggplot(stem_gene_long, aes(x = condition, y = expr)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  facet_wrap(~gene, scales = "free_y") +
  theme_bw()

stem_gene_tests <- stem_gene_long %>%
  dplyr::group_by(gene) %>%
  dplyr::summarise(
    disomy_mean = mean(expr[condition == "disomy"]),
    ts21_mean   = mean(expr[condition == "Ts21"]),
    delta       = ts21_mean - disomy_mean,
    p_wilcox    = wilcox.test(expr ~ condition, exact = FALSE)$p.value,
    .groups = "drop"
  )

stem_gene_donor
stem_gene_tests

In [ ]:
# ------------------------------------------------------------
# C) donor-pseudobulk stem_core RNA
# ------------------------------------------------------------
stem_rna <- subset(rna, subset = stem_core == "stem_core" & !is.na(donor))

pb_stem <- AggregateExpression(
  object = stem_rna,
  group.by = "donor",
  assays = "RNA",
  return.seurat = TRUE
)

pb_genes <- c(
  "EBF1", "PAX5", "CD24", "DNTT",
  "GATA1", "KLF1", "ALAS2", "AHSP", "HBB",
  "IFI6", "IFIT1", "ISG15", "MX1"
)

pb_mat <- GetAssayData(pb_stem, assay = "RNA", layer = "data")[pb_genes, , drop = FALSE]
pb_mat

In [ ]:

# ------------------------------------------------------------
# D) donor-aware B-lineage depletion
# ------------------------------------------------------------
meta <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(!is.na(donor), !is.na(condition), !is.na(celltype_rna)) %>%
  tibble::as_tibble()

# define B-lineage states present in your object
b_states <- intersect(
  c("B_primed_prog", "proB_preB", "preB"),
  unique(meta$celltype_rna)
)

# define hematopoietic denominator
exclude_nonheme <- c("hepatocyte_like", "unresolved_contam_like", "low_complexity")
heme_meta <- meta %>%
  dplyr::filter(!celltype_rna %in% exclude_nonheme)

# denominator 1: fraction of all hematopoietic cells
b_all <- heme_meta %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_total = length(cellNames),
    n_B = sum(celltype_rna %in% b_states, na.rm = TRUE),
    frac_B_all_heme = n_B / n_total,
    .groups = "drop"
  )

b_all

ggplot(b_all, aes(x = condition, y = frac_B_all_heme)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  theme_bw()

# denominator 2: fraction within progenitor-focused compartment
prog_states <- intersect(
  c("HSC_MPP", "LMPP", "B_primed_prog", "proB_preB", "ery_prog",
    "erythroid_like_2", "myeloid_DC_like", "granulocyte_prog", "mixed_unresolved"),
  unique(meta$celltype_rna)
)

prog_meta <- meta %>%
  dplyr::filter(celltype_rna %in% prog_states)

b_prog <- prog_meta %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_prog = length(cellNames),
    n_B = sum(celltype_rna %in% b_states, na.rm = TRUE),
    frac_B_within_prog = n_B / n_prog,
    .groups = "drop"
  )

b_prog

ggplot(b_prog, aes(x = condition, y = frac_B_within_prog)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  theme_bw()

wilcox.test(frac_B_within_prog ~ condition, data = b_prog, exact = FALSE)


In [ ]:
# ------------------------------------------------------------
# Narrow progenitor refinement: keep only plausible upstream/progenitor states
# ------------------------------------------------------------
table(rna$celltype_rna, useNA = "ifany")

keep_states <- intersect(
  c(
    "HSC_MPP",
    "B_primed_prog",
    "erythroid_like_2",
    "mixed_unresolved"
  ),
  unique(rna$celltype_rna)
)

keep_states

Idents(rna) <- "celltype_rna"
rna_prog2 <- subset(rna, idents = keep_states)

table(rna_prog2$celltype_rna, useNA = "ifany")
DimPlot(rna_prog2, reduction = "umap", group.by = "celltype_rna", label = TRUE, raster = TRUE)
DimPlot(rna_prog2, reduction = "umap", group.by = "condition", raster = TRUE)
DimPlot(rna_prog2, reduction = "umap", group.by = "donor", raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# Reprocess the narrow progenitor subset
# ------------------------------------------------------------
DefaultAssay(rna_prog2) <- "RNA"

rna_prog2 <- JoinLayers(rna_prog2, assay = "RNA")

rna_prog2 <- NormalizeData(rna_prog2, verbose = FALSE)
rna_prog2 <- FindVariableFeatures(rna_prog2, verbose = FALSE)
rna_prog2 <- ScaleData(rna_prog2, verbose = FALSE)
rna_prog2 <- RunPCA(rna_prog2, verbose = FALSE)

rna_prog2 <- RunHarmony(
  object = rna_prog2,
  group.by.vars = "sample",
  reduction.use = "pca",
  dims.use = 1:30,
  verbose = FALSE
)

rna_prog2 <- RunUMAP(rna_prog2, reduction = "harmony", dims = 1:30, verbose = FALSE)
rna_prog2 <- FindNeighbors(rna_prog2, reduction = "harmony", dims = 1:30, verbose = FALSE)
rna_prog2 <- FindClusters(rna_prog2, resolution = 0.2, verbose = FALSE)

DimPlot(rna_prog2, reduction = "umap", group.by = "seurat_clusters", label = TRUE, raster = TRUE)
DimPlot(rna_prog2, reduction = "umap", group.by = "condition", raster = TRUE)
DimPlot(rna_prog2, reduction = "umap", group.by = "donor", raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# Markers in the narrow progenitor subset
# ------------------------------------------------------------
Idents(rna_prog2) <- "seurat_clusters"

prog2_markers <- FindAllMarkers(
  object = rna_prog2,
  assay = "RNA",
  only.pos = TRUE,
  test.use = "wilcox",
  min.pct = 0.10,
  logfc.threshold = 0.25,
  return.thresh = 0.05
)

prog2_top10 <- prog2_markers %>%
  dplyr::group_by(cluster) %>%
  dplyr::slice_max(order_by = avg_log2FC, n = 10, with_ties = FALSE) %>%
  dplyr::ungroup()

print(prog2_top10, n = 200)

In [ ]:
# ------------------------------------------------------------
# Program scores to resolve B-priming vs erythroid vs IFN
# ------------------------------------------------------------
b_priming_genes <- list(c("EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A"))
ery_heme_genes  <- list(c("GATA1","KLF1","ALAS2","AHSP","HBA1","HBA2","HBB","HMBS","FECH"))
ifn_genes       <- list(c("IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","IFI44L"))
stem_genes      <- list(c("CD34","SPINK2","HLF","FGD5","PROM1","FLT3","GATA2","MECOM"))

rna_prog2 <- AddModuleScore(rna_prog2, features = b_priming_genes, name = "Bpriming")
rna_prog2 <- AddModuleScore(rna_prog2, features = ery_heme_genes,  name = "EryHeme")
rna_prog2 <- AddModuleScore(rna_prog2, features = ifn_genes,       name = "IFN")
rna_prog2 <- AddModuleScore(rna_prog2, features = stem_genes,      name = "StemCoreProg")

DotPlot(
  rna_prog2,
  features = c(
    "CD34","SPINK2","HLF","FGD5","PROM1","FLT3","GATA2","MECOM",
    "EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A",
    "GATA1","KLF1","ALAS2","AHSP","HBB",
    "IFI6","IFIT1","ISG15","MX1"
  ),
  group.by = "seurat_clusters"
) + RotatedAxis()

FeaturePlot(
  rna_prog2,
  features = c("StemCoreProg1","Bpriming1","EryHeme1","IFN1"),
  reduction = "umap",
  raster = TRUE,
  ncol = 2
)

In [ ]:
# ------------------------------------------------------------
# Real mapping after inspection of prog2_top10 + DotPlot
# ------------------------------------------------------------
cluster_to_celltype_prog2 <- c(
  "0" = "B_primed_prog",
  "1" = "HSC_MPP",
  "2" = "erythroid_prog",
  "3" = "erythroid_heme_prog",
  "4" = "NK_like_leak",
  "5" = "mixed_unresolved"
)

rna_prog2$celltype_refined2 <- unname(
  cluster_to_celltype_prog2[as.character(rna_prog2$seurat_clusters)]
)

# strict real stem_core
stem_core_labels2 <- c("HSC_MPP")

rna_prog2$stem_core_refined2 <- ifelse(
  rna_prog2$celltype_refined2 %in% stem_core_labels2,
  "stem_core",
  "other"
)

table(rna_prog2$celltype_refined2, useNA = "ifany")
table(rna_prog2$stem_core_refined2, useNA = "ifany")

DimPlot(rna_prog2, reduction = "umap", group.by = "celltype_refined2", label = TRUE, raster = TRUE)

In [ ]:
# ------------------------------------------------------------
# Write refined narrow-progenitor labels back into full rna object
# ------------------------------------------------------------
rna$celltype_rna <- as.character(rna$broad_annot)
rna$stem_core <- "other"

rna$celltype_rna[colnames(rna_prog2)] <- rna_prog2$celltype_refined2
rna$stem_core[colnames(rna_prog2)] <- rna_prog2$stem_core_refined2

rna$stem_core <- factor(rna$stem_core, levels = c("stem_core", "other"))

table(rna$celltype_rna, useNA = "ifany")
table(rna$stem_core, useNA = "ifany")

DimPlot(rna, reduction = "umap", group.by = "celltype_rna", label = TRUE, raster = TRUE)

In [ ]:
# start from the full Seurat object
rna$celltype_plot <- as.character(rna$celltype_rna)

# remove labels you do not want in the broad UMAP
rna$celltype_plot[rna$celltype_plot %in% c(
  "NK_like_leak",
  "mixed_unresolved",
  "unassigned",
  "unresolved_contam_like"
)] <- NA

# keep only cells with retained labels
cells_keep <- rownames(rna[[]])[!is.na(rna$celltype_plot)]
rna_plot <- subset(rna, cells = cells_keep)

table(rna_plot$celltype_plot, useNA = "ifany")

library(Seurat)
library(ggplot2)

# recreate rna_plot
rna$celltype_plot <- as.character(rna$celltype_rna)

rna$celltype_plot[rna$celltype_plot %in% c(
  "NK_like_leak",
  "mixed_unresolved",
  "unassigned",
  "unresolved_contam_like"
)] <- NA

cells_keep <- rownames(rna[[]])[!is.na(rna$celltype_plot)]
rna_plot <- subset(rna, cells = cells_keep)

shared_cols <- c(
  "B_prog"   = "#F8766D",
  "Ery_heme" = "#B79F00",
  "Ery_like" = "#00BA38",
  "Ery_prog" = "#00BFC4",
  "HSC_MPP"  = "#619CFF",
  "NK"       = "#F564E3"
)

rna_plot$celltype_plot2 <- as.character(rna_plot$celltype_plot)

rna_plot$celltype_plot2[rna_plot$celltype_plot2 == "B_primed_prog"] <- "B Prog"
rna_plot$celltype_plot2[rna_plot$celltype_plot2 == "erythroid_heme_prog"] <- "Ery Heme"
rna_plot$celltype_plot2[rna_plot$celltype_plot2 == "erythroid_like"] <- "Ery-like"
rna_plot$celltype_plot2[rna_plot$celltype_plot2 == "erythroid_prog"] <- "Ery Prog"
rna_plot$celltype_plot2[rna_plot$celltype_plot2 == "HSC_MPP"] <- "HSC MPP"


rna_plot$celltype_plot2 <- factor(
  rna_plot$celltype_plot2,
  levels = c("B Prog", "Ery Heme", "Ery-like", "Ery Prog", "HSC MPP", "NK")
)

p_broad_clean <- DimPlot(
  rna_plot,
  reduction = "umap",
  group.by = "celltype_plot2",
  cols = c(
    "B Prog"   = "#F8766D",
    "Ery Heme" = "#B79F00",
    "Ery-like" = "#00BA38",
    "Ery Prog" = "#00BFC4",
    "HSC MPP"  = "#619CFF",
    "NK"       = "#F564E3"
  ),
  label = FALSE,
  raster = TRUE
) +
  coord_fixed() +
  theme_classic() +
  labs(
    title = NULL,
    x = "UMAP1",
    y = "UMAP2"
  ) +
  theme(
    plot.title = element_blank(),
    legend.position = "right",
    axis.text = element_blank(),
    axis.ticks = element_blank()
  )

print(p_broad_clean)

outdir <- "/path/to/nature_study/RNA_final_results"

ggsave(
  filename = file.path(outdir, "UMAP_broad_clean_shortlabels.pdf"),
  plot = p_broad_clean,
  width = 6,
  height = 5,
  device = cairo_pdf
)

ggsave(
  filename = file.path(outdir, "UMAP_broad_clean_shortlabels.tiff"),
  plot = p_broad_clean,
  width = 6,
  height = 5,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
# ------------------------------------------------------------
# Update full-rna module scores without overwriting refined labels
# ------------------------------------------------------------
DefaultAssay(rna) <- "RNA"

b_priming_genes <- list(c("EBF1","PAX5","CD24","DNTT","VPREB1","IGLL1","CD79A"))
ery_heme_genes  <- list(c("GATA1","KLF1","ALAS2","AHSP","HBA1","HBA2","HBB","HMBS","FECH"))
ifn_genes       <- list(c("IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","IFI44L"))

# remove old scores if present
for (nm in c("Bpriming1", "EryHeme1", "IFN1")) {
  if (nm %in% colnames(rna[[]])) {
    rna[[nm]] <- NULL
  }
}

rna <- AddModuleScore(rna, features = b_priming_genes, name = "Bpriming")
rna <- AddModuleScore(rna, features = ery_heme_genes,  name = "EryHeme")
rna <- AddModuleScore(rna, features = ifn_genes,       name = "IFN")

# sanity checks
table(rna$celltype_rna, useNA = "ifany")
table(rna$stem_core, useNA = "ifany")
summary(rna$Bpriming1)
summary(rna$EryHeme1)
summary(rna$IFN1)

In [ ]:
# ------------------------------------------------------------
# A) donor-aware program shift inside stem_core
# ------------------------------------------------------------
stopifnot(all(c(
  "celltype_rna", "stem_core", "donor", "condition",
  "Bpriming1", "EryHeme1", "IFN1"
) %in% colnames(rna[[]])))

stem_df <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(
    stem_core == "stem_core",
    !is.na(donor),
    !is.na(condition)
  ) %>%
  tibble::as_tibble()

# check how many stem_core cells per donor
stem_counts <- stem_df %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_stem_core_cells = dplyr::n(),
    .groups = "drop"
  )

stem_counts

# donor-level mean program scores
stem_prog <- stem_df %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_cells   = dplyr::n(),
    Bpriming1 = mean(Bpriming1, na.rm = TRUE),
    EryHeme1  = mean(EryHeme1,  na.rm = TRUE),
    IFN1      = mean(IFN1,      na.rm = TRUE),
    .groups = "drop"
  )

stem_prog

stem_prog_long <- stem_prog %>%
  tidyr::pivot_longer(
    cols = c(Bpriming1, EryHeme1, IFN1),
    names_to = "program",
    values_to = "score"
  )

ggplot(stem_prog_long, aes(x = condition, y = score)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  facet_wrap(~program, scales = "free_y") +
  theme_bw()

stem_prog_tests <- stem_prog_long %>%
  dplyr::group_by(program) %>%
  dplyr::summarise(
    disomy_mean = mean(score[condition == "disomy"]),
    ts21_mean   = mean(score[condition == "Ts21"]),
    delta       = ts21_mean - disomy_mean,
    p_wilcox    = wilcox.test(score ~ condition, exact = FALSE)$p.value,
    .groups = "drop"
  )

stem_prog_tests

In [ ]:
# ------------------------------------------------------------
# B) donor-aware gene-level check inside stem_core
# ------------------------------------------------------------
genes_b   <- c("EBF1", "PAX5", "CD24", "DNTT")
genes_ery <- c("GATA1", "KLF1", "ALAS2", "AHSP", "HBB")
genes_ifn <- c("IFI6", "IFIT1", "ISG15", "MX1")
genes_use <- c(genes_b, genes_ery, genes_ifn)

stem_expr <- FetchData(
  object = rna,
  vars = c("stem_core", "donor", "condition", genes_use)
) %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(stem_core == "stem_core") %>%
  tibble::as_tibble()

stem_gene_donor <- stem_expr %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    dplyr::across(dplyr::all_of(genes_use), ~mean(.x, na.rm = TRUE)),
    .groups = "drop"
  )

stem_gene_long <- stem_gene_donor %>%
  tidyr::pivot_longer(
    cols = dplyr::all_of(genes_use),
    names_to = "gene",
    values_to = "expr"
  )

ggplot(stem_gene_long, aes(x = condition, y = expr)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  facet_wrap(~gene, scales = "free_y") +
  theme_bw()

stem_gene_tests <- stem_gene_long %>%
  dplyr::group_by(gene) %>%
  dplyr::summarise(
    disomy_mean = mean(expr[condition == "disomy"]),
    ts21_mean   = mean(expr[condition == "Ts21"]),
    delta       = ts21_mean - disomy_mean,
    p_wilcox    = wilcox.test(expr ~ condition, exact = FALSE)$p.value,
    .groups = "drop"
  )

stem_gene_donor
stem_gene_tests

In [ ]:
# ------------------------------------------------------------
# C) donor-pseudobulk stem_core RNA
# ------------------------------------------------------------
stem_rna <- subset(rna, subset = stem_core == "stem_core" & !is.na(donor))

pb_stem <- AggregateExpression(
  object = stem_rna,
  group.by = "donor",
  assays = "RNA",
  return.seurat = TRUE
)

pb_genes <- c(
  "EBF1", "PAX5", "CD24", "DNTT",
  "GATA1", "KLF1", "ALAS2", "AHSP", "HBB",
  "IFI6", "IFIT1", "ISG15", "MX1"
)

pb_mat <- GetAssayData(pb_stem, assay = "RNA", layer = "data")[pb_genes, , drop = FALSE]
pb_mat

In [ ]:
# ------------------------------------------------------------
# D) donor-aware B-lineage depletion
# ------------------------------------------------------------
meta <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(!is.na(donor), !is.na(condition), !is.na(celltype_rna)) %>%
  tibble::as_tibble()

# define B-lineage states present in your object
b_states <- intersect(
  c("B_primed_prog", "proB_preB", "preB"),
  unique(meta$celltype_rna)
)

# define hematopoietic denominator
exclude_nonheme <- c("hepatocyte_like", "unresolved_contam_like", "low_complexity")
heme_meta <- meta %>%
  dplyr::filter(!celltype_rna %in% exclude_nonheme)

# denominator 1: fraction of all hematopoietic cells
b_all <- heme_meta %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_total = length(cellNames),
    n_B = sum(celltype_rna %in% b_states, na.rm = TRUE),
    frac_B_all_heme = n_B / n_total,
    .groups = "drop"
  )

b_all

ggplot(b_all, aes(x = condition, y = frac_B_all_heme)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  theme_bw()

# denominator 2: fraction within progenitor-focused compartment
prog_states

prog_meta <- meta %>%
  dplyr::filter(celltype_rna %in% prog_states)

b_prog <- prog_meta %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_prog = length(cellNames),
    n_B = sum(celltype_rna %in% b_states, na.rm = TRUE),
    frac_B_within_prog = n_B / n_prog,
    .groups = "drop"
  )

b_prog

ggplot(b_prog, aes(x = condition, y = frac_B_within_prog)) +
  geom_boxplot(outlier.shape = NA) +
  geom_point(aes(shape = donor), size = 3, position = position_jitter(width = 0.06)) +
  theme_bw()

wilcox.test(frac_B_within_prog ~ condition, data = b_prog, exact = FALSE)


In [ ]:
# exact denominator used in the stats notebook
states_keep <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")

meta <- rna[[]]
cells_keep <- rownames(meta)[meta$celltype_rna %in% states_keep]

rna_prog_match <- subset(rna, cells = cells_keep)

# shortened labels for plotting
rna_prog_match$celltype_plot2 <- as.character(rna_prog_match$celltype_rna)
rna_prog_match$celltype_plot2[rna_prog_match$celltype_plot2 == "B_primed_prog"] <- "B_prog"
rna_prog_match$celltype_plot2[rna_prog_match$celltype_plot2 == "erythroid_heme_prog"] <- "Ery_heme"
rna_prog_match$celltype_plot2[rna_prog_match$celltype_plot2 == "erythroid_prog"] <- "Ery_prog"

rna_prog_match$celltype_plot2 <- factor(
  rna_prog_match$celltype_plot2,
  levels = c("B_prog", "Ery_heme", "Ery_prog", "HSC_MPP")
)

# same colors as the broad UMAP
shared_cols <- c(
  "B_prog"   = "#F8766D",
  "Ery_heme" = "#B79F00",
  "Ery_like" = "#00BA38",
  "Ery_prog" = "#00BFC4",
  "HSC_MPP"  = "#619CFF",
  "NK"       = "#F564E3"
)

p_umap_match <- DimPlot(
  rna_prog_match,
  reduction = "umap",
  group.by = "celltype_plot2",
  cols = shared_cols[c("B_prog", "Ery_heme", "Ery_prog", "HSC_MPP")],
  label = FALSE,
  raster = TRUE
) +
  coord_fixed() +
  theme_classic() +
  theme(
    legend.position = "right",
    axis.text = element_blank(),
    axis.ticks = element_blank()
  ) +
  labs(
    x = "UMAP1",
    y = "UMAP2"
  )

print(p_umap_match)

outdir <- "/path/to/nature_study/RNA_final_results"

ggsave(
  filename = file.path(outdir, "UMAP_refined4_clean_shortlabels_matchedcolors.pdf"),
  plot = p_umap_match,
  width = 6,
  height = 5,
  device = cairo_pdf
)

ggsave(
  filename = file.path(outdir, "UMAP_refined4_clean_shortlabels_matchedcolors.tiff"),
  plot = p_umap_match,
  width = 6,
  height = 5,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
library(Seurat)
library(ggplot2)

# exact denominator used in the stats notebook
states_keep <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")

meta <- rna[[]]
cells_keep <- rownames(meta)[meta$celltype_rna %in% states_keep]

rna_prog_match <- subset(rna, cells = cells_keep)

# keep original labels/order
rna_prog_match$condition <- factor(rna_prog_match$condition, levels = c("disomy", "Ts21"))

p_umap_condition <- DimPlot(
  rna_prog_match,
  reduction = "umap",
  group.by = "condition",
  label = FALSE,
  raster = TRUE
) +
  coord_fixed() +
  theme_classic() +
  theme(
    legend.position = "right",
    axis.text = element_blank(),
    axis.ticks = element_blank()
  ) +
  labs(
    x = "UMAP1",
    y = "UMAP2"
  )

print(p_umap_condition)

outdir <- "/path/to/nature_study/RNA_final_results"

ggsave(
  filename = file.path(outdir, "UMAP_refined4_condition_clean.pdf"),
  plot = p_umap_condition,
  width = 6,
  height = 5,
  device = cairo_pdf
)

ggsave(
  filename = file.path(outdir, "UMAP_refined4_condition_clean.tiff"),
  plot = p_umap_condition,
  width = 6,
  height = 5,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
library(Seurat)
library(ggplot2)

# exact denominator used in the stats notebook
states_keep <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")

meta <- rna[[]]
cells_keep <- rownames(meta)[meta$celltype_rna %in% states_keep]

rna_prog_match <- subset(rna, cells = cells_keep)

# plotting labels
rna_prog_match$condition_plot <- as.character(rna_prog_match$condition)
rna_prog_match$condition_plot[rna_prog_match$condition_plot == "disomy"] <- "D"
rna_prog_match$condition_plot <- factor(rna_prog_match$condition_plot, levels = c("D", "Ts21"))

p_umap_condition <- DimPlot(
  rna_prog_match,
  reduction = "umap",
  group.by = "condition_plot",
  label = FALSE,
  raster = TRUE
) +
  coord_fixed() +
  theme_classic() +
  theme(
    legend.position = "right",
    axis.text = element_blank(),
    axis.ticks = element_blank()
  ) +
  labs(
    title = NULL,
    x = "UMAP1",
    y = "UMAP2"
  )

print(p_umap_condition)

outdir <- "/path/to/nature_study/RNA_final_results"

ggsave(
  filename = file.path(outdir, "UMAP_refined4_condition_clean.pdf"),
  plot = p_umap_condition,
  width = 6,
  height = 5,
  device = cairo_pdf
)

ggsave(
  filename = file.path(outdir, "UMAP_refined4_condition_clean.tiff"),
  plot = p_umap_condition,
  width = 6,
  height = 5,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
# ------------------------------------------------------------
# D) donor-aware B-lineage depletion
# Updated to match the restricted stats-notebook definition
# ------------------------------------------------------------
library(dplyr)
library(tibble)
library(ggplot2)

meta <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::filter(!is.na(donor), !is.na(condition), !is.na(celltype_rna)) %>%
  tibble::as_tibble()

# exact denominator used in the stats notebook
prog_states <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")

prog_meta <- meta %>%
  dplyr::filter(celltype_rna %in% prog_states)

# donor-level restricted composition table
b_prog <- prog_meta %>%
  dplyr::group_by(donor, condition) %>%
  dplyr::summarise(
    n_prog = length(cellNames),
    n_B = sum(celltype_rna == "B_primed_prog", na.rm = TRUE),
    frac_B_within_prog = n_B / n_prog,
    .groups = "drop"
  ) %>%
  dplyr::mutate(
    condition = factor(condition, levels = c("disomy", "Ts21")),
    group_shape = ifelse(condition == "disomy", "D", "Ts21"),
    group_shape = factor(group_shape, levels = c("D", "Ts21"))
  ) %>%
  dplyr::arrange(condition, donor)

b_prog

# Wilcoxon test
wilcox_res <- wilcox.test(frac_B_within_prog ~ condition, data = b_prog, exact = FALSE)
wilcox_res

# Plot
p_bprog <- ggplot(b_prog, aes(x = condition, y = frac_B_within_prog)) +
  geom_boxplot(
    aes(fill = condition),
    outlier.shape = NA,
    width = 0.55,
    alpha = 0.45,
    color = "black"
  ) +
  geom_point(
    aes(shape = group_shape),
    color = "black",
    size = 3,
    position = position_jitter(width = 0.06, height = 0),
    show.legend = FALSE
  ) +
  scale_shape_manual(values = c("D" = 16, "Ts21" = 17)) +
  scale_x_discrete(labels = c("disomy" = "D", "Ts21" = "Ts21")) +
  scale_fill_manual(values = c("disomy" = "#4C78A8", "Ts21" = "#E45756")) +
  theme_bw() +
  theme(
    panel.grid.minor = element_blank(),
    axis.title.x = element_blank(),
    legend.position = "none"
  ) +
  labs(
    y = "Fraction B_prog within refined progenitors"
  )

print(p_bprog)

outdir <- "/path/to/nature_study/RNA_final_results"

write.table(
  b_prog,
  file = file.path(outdir, "b_prog_refined_match_stats.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot.pdf"),
  plot = p_bprog,
  width = 4.2,
  height = 4
)

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot.tiff"),
  plot = p_bprog,
  width = 4.2,
  height = 4,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:

stats_file <- "/path/to/nature_study/RNA_export_for_stats/progenitor_donor_counts_for_composition.tsv"
prog_counts <- read.delim(stats_file, check.names = FALSE)

prog_counts$condition <- factor(prog_counts$condition, levels = c("disomy", "Ts21"))
prog_counts$group_shape <- ifelse(prog_counts$condition == "disomy", "D", "Ts21")
prog_counts$group_shape <- factor(prog_counts$group_shape, levels = c("D", "Ts21"))

p_bprog <- ggplot(prog_counts, aes(x = condition, y = frac_B)) +
  geom_boxplot(
    aes(fill = condition),
    outlier.shape = NA,
    width = 0.55,
    alpha = 0.45,
    color = "black"
  ) +
  geom_point(
    aes(shape = group_shape),
    color = "black",
    size = 3,
    position = position_jitter(width = 0.06, height = 0),
    show.legend = FALSE
  ) +
  scale_shape_manual(values = c("D" = 16, "Ts21" = 17)) +
  scale_x_discrete(labels = c("disomy" = "D", "Ts21" = "Ts21")) +
  scale_fill_manual(values = c("disomy" = "#4C78A8", "Ts21" = "#E45756")) +
  theme_classic() +
  theme(
    axis.title.x = element_blank(),
    legend.position = "none"
  ) +
  labs(
    y = "B-primed Progenitor Fraction"
  )

print(p_bprog)

outdir <- "/path/to/nature_study/RNA_final_results"

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot_clean.pdf"),
  plot = p_bprog,
  width = 4.2,
  height = 4
)

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot_clean.tiff"),
  plot = p_bprog,
  width = 4.2,
  height = 4,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
p_bprog <- ggplot(prog_counts, aes(x = condition, y = frac_B)) +
  geom_boxplot(
    aes(fill = condition),
    outlier.shape = NA,
    width = 0.55,
    alpha = 0.45,
    color = "black"
  ) +
  geom_point(
    aes(shape = group_shape),
    color = "black",
    size = 3,
    position = position_jitter(width = 0.06, height = 0),
    show.legend = FALSE
  ) +
  scale_shape_manual(values = c("D" = 16, "Ts21" = 17)) +
  scale_x_discrete(labels = c("disomy" = "D", "Ts21" = "Ts21")) +
  scale_fill_manual(values = c("disomy" = "#4C78A8", "Ts21" = "#E45756")) +
  theme_classic() +
  theme(
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 14),
    axis.title.y = element_text(size = 14),
    legend.position = "none",
    plot.subtitle = element_text(size = 11, hjust = 0.5)
  ) +
  labs(
    y = "B Progenitor Fraction",
    subtitle = paste(
      "Quasibinomial P = 0.054",
      "Mixed-model P = 0.0019, OR = 0.10",
      sep = "\n"
    )
  )

print(p_bprog)

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot_clean.pdf"),
  plot = p_bprog,
  width = 4.2,
  height = 4
)

ggsave(
  filename = file.path(outdir, "B_prog_fraction_refined_boxplot_clean.tiff"),
  plot = p_bprog,
  width = 4.2,
  height = 4,
  units = "in",
  dpi = 600,
  compression = "lzw"
)

In [ ]:
# ------------------------------------------------------------
# Save final refined RNA results
# ------------------------------------------------------------

outdir <- "/path/to/nature_study/RNA_final_results"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# save Seurat objects
saveRDS(rna,       file.path(outdir, "rna_full_final.rds"))
saveRDS(rna_prog2, file.path(outdir, "rna_prog2_final.rds"))

# full annotation table
anno_final <- as.data.frame(rna[[]]) %>%
  tibble::rownames_to_column("cellNames") %>%
  dplyr::transmute(
    cellNames,
    sample,
    donor,
    condition,
    seurat_clusters = as.character(seurat_clusters),
    broad_annot = as.character(broad_annot),
    rna_celltype = as.character(celltype_rna),
    stem_core = as.character(stem_core),
    Bpriming1,
    EryHeme1,
    IFN1
  )

write.table(
  anno_final,
  file.path(outdir, "rna_annotations_final.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

# progenitor refinement markers
if (exists("prog2_markers")) {
  write.table(
    prog2_markers,
    file.path(outdir, "prog2_markers.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("prog2_top10")) {
  write.table(
    prog2_top10,
    file.path(outdir, "prog2_top10.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

# stem_core donor-level summaries
if (exists("stem_counts")) {
  write.table(
    stem_counts,
    file.path(outdir, "stem_counts.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("stem_prog")) {
  write.table(
    stem_prog,
    file.path(outdir, "stem_prog.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("stem_prog_tests")) {
  write.table(
    stem_prog_tests,
    file.path(outdir, "stem_prog_tests.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("stem_gene_donor")) {
  write.table(
    stem_gene_donor,
    file.path(outdir, "stem_gene_donor.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("stem_gene_tests")) {
  write.table(
    stem_gene_tests,
    file.path(outdir, "stem_gene_tests.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

# B-lineage depletion summaries
if (exists("b_all")) {
  write.table(
    b_all,
    file.path(outdir, "b_all.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

if (exists("b_prog")) {
  write.table(
    b_prog,
    file.path(outdir, "b_prog.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}

cat("Saved RNA results to:", outdir, "\n")

In [ ]:
archr_proj <- saveArchRProject(
  ArchRProj = archr_proj,
  outputDirectory = "/path/to/nature_study/ArchR/ArchRProject_finalRNA",
  load = TRUE
)